# Laboratorio 4 - Resposta em Frequencia e Equacao de Diferencas

Este notebook implementa o sistema do Laboratorio 4, plota a resposta em frequencia, escreve a equacao de diferencas, filtra um sinal escolhido e mostra a resposta impulsiva.

Funcao de transferencia dada:

$$H(z) = \frac{5z}{z^2 - 1.3z + 0.4}$$

Dividindo numerador e denominador por $z^2$:

$$H(z) = \frac{5z^{-1}}{1 - 1.3z^{-1} + 0.4z^{-2}}$$

Assim, para usar `scipy.signal.freqz` e `scipy.signal.lfilter`:

`b = [0, 5, 0]` e `a = [1, -1.3, 0.4]`.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

b = np.array([0.0, 5.0, 0.0])
a = np.array([1.0, -1.3, 0.4])

print(f"Coeficientes do numerador b: {b}")
print(f"Coeficientes do denominador a: {a}")


## Equacao de diferencas

A partir de

$$H(z) = \frac{Y(z)}{X(z)} = \frac{5z^{-1}}{1 - 1.3z^{-1} + 0.4z^{-2}}$$

temos:

$$Y(z)(1 - 1.3z^{-1} + 0.4z^{-2}) = X(z)(5z^{-1})$$

Logo:

$$y[n] - 1.3y[n-1] + 0.4y[n-2] = 5x[n-1]$$

ou, isolando a saida atual:

$$y[n] = 1.3y[n-1] - 0.4y[n-2] + 5x[n-1]$$


## Resposta em frequencia

O modulo mostra quanto cada frequencia e amplificada ou atenuada. A fase mostra o deslocamento angular aplicado a cada componente de frequencia.

In [ ]:
w, h = signal.freqz(b, a, worN=2048)

magnitude = np.abs(h)
phase = np.unwrap(np.angle(h))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(w / np.pi, magnitude)
axes[0].set_title("Resposta em frequencia - modulo")
axes[0].set_ylabel("|H(e^jw)|")
axes[0].grid(True)

axes[1].plot(w / np.pi, phase)
axes[1].set_title("Resposta em frequencia - fase")
axes[1].set_xlabel("Frequencia normalizada (x pi rad/amostra)")
axes[1].set_ylabel("Fase (rad)")
axes[1].grid(True)

plt.tight_layout()
plt.show()

poles = np.roots(a)
zeros = np.roots(b)

print(f"Polos: {poles}")
print(f"Zeros: {zeros}")
print(f"Ganho em DC aproximado: {magnitude[0]:.2f}")
print(f"Modulo em pi rad/amostra aproximado: {magnitude[-1]:.2f}")


## Analise da resposta em frequencia

Os polos do sistema ficam em `0.8` e `0.5`, ambos dentro do circulo unitario, portanto o filtro e estavel. Como existe um polo relativamente proximo de `z = 1`, o ganho nas baixas frequencias fica alto. Assim, o sistema amplifica bastante componentes lentas do sinal e amplifica menos as componentes proximas de alta frequencia.

## Filtragem de um sinal escolhido

O sinal escolhido foi uma soma de duas senoides: uma componente de baixa frequencia e uma componente de alta frequencia. A filtragem foi feita com `signal.lfilter(b, a, x)`, que aplica diretamente a equacao de diferencas do sistema.

In [ ]:
n = np.arange(200)

low_frequency = 0.10 * np.sin(0.08 * np.pi * n)
high_frequency = 0.40 * np.sin(0.70 * np.pi * n)
x = low_frequency + high_frequency

y = signal.lfilter(b, a, x)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(n, x, label="Entrada x[n]")
axes[0].set_title("Sinal de entrada")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(n, y, color="tab:orange", label="Saida y[n]")
axes[1].set_title("Sinal filtrado")
axes[1].set_xlabel("n")
axes[1].set_ylabel("Amplitude")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
def single_sided_spectrum(samples: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    spectrum = np.fft.rfft(samples)
    frequency = np.fft.rfftfreq(len(samples), d=1.0)
    magnitude = np.abs(spectrum) / len(samples)
    return frequency, magnitude


freq_x, mag_x = single_sided_spectrum(x)
freq_y, mag_y = single_sided_spectrum(y)

plt.figure(figsize=(10, 5))
plt.plot(freq_x * 2, mag_x, label="Entrada")
plt.plot(freq_y * 2, mag_y, label="Saida")
plt.title("Comparacao espectral antes e depois da filtragem")
plt.xlabel("Frequencia normalizada (x pi rad/amostra)")
plt.ylabel("Magnitude")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


## Analise da filtragem

Como o modulo da resposta em frequencia e maior nas baixas frequencias, a senoide lenta aparece bem mais forte na saida. A senoide de alta frequencia continua existindo, mas fica proporcionalmente menos destacada. Isso confirma o comportamento observado no grafico de modulo.

## Resposta impulsiva

A resposta impulsiva mostra a saida do sistema quando a entrada e um impulso unitario. Como o sistema e IIR, a resposta nao acaba imediatamente, mas decai ao longo das amostras porque os polos estao dentro do circulo unitario.

In [ ]:
# Forma polinomial em z equivalente a H(z) = 5z / (z^2 - 1.3z + 0.4).
b_transfer_function = np.array([5.0, 0.0])
system = signal.dlti(b_transfer_function, a, dt=1)
t_impulse, impulse_response = signal.dimpulse(system, n=80)
h_impulse = np.squeeze(impulse_response)

plt.figure(figsize=(10, 5))
plt.stem(np.squeeze(t_impulse), h_impulse)
plt.title("Resposta impulsiva do sistema")
plt.xlabel("n")
plt.ylabel("h[n]")
plt.grid(True)
plt.tight_layout()
plt.show()
